In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_17_brownian_langevin_and_dpd.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 17: Brownian Dynamics, Langevin Equations, and Dissipative Particle Dynamics

_Session 012b30b8 · Estimated time: ~125 minutes · 40 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Place Path-Integral MC, Ab Initio MD, Classical All-Atom MD, Langevin Dynamics, Brownian Dynamics, DPD, Metropolis MC, and Kinetic MC along the accuracy-versus-timescale continuum.
- Explain why eliminating explicit solvent molecules saves >90% of CPU time in colloidal, polymeric, and biomolecular simulations and when Monte Carlo moves yield physically realistic dynamics.
- Derive the Smoluchowski advection-diffusion equation, prove that its zero-flux stationary state is the canonical Boltzmann distribution, and evaluate its short-time drift and variance moments (including the divergence correction ∇·D).
- Apply Trotter operator splitting to combine the Gaussian diffusion Green's function with deterministic drift, and explain how Hybrid Brownian / Smart Monte Carlo guarantees exact canonical sampling.
- Formulate hydrodynamic interactions via the Oseen tensor (∝ 1/r) and evaluate the O(N^3) computational cost of Cholesky noise factorization.
- Analyze the Langevin equation across inertial/ballistic (t ≪ ζ^-1) and overdamped Brownian (t ≫ ζ^-1) regimes using the fluctuation-dissipation theorem.
- Explain why global Langevin and Andersen thermostats destroy local momentum conservation and Navier-Stokes hydrodynamics, and how Dissipative Particle Dynamics (DPD) restores Galilean invariance and hydrodynamic flow via pairwise central forces.


In [2]:
ws.render_section('sec_multiscale_continuum_and_mc_dynamics')

### The Accuracy vs. Speed Continuum of Simulation Methods

Atomic-scale and mesoscale simulations span more than twelve orders of magnitude in length and time. To reach longer timescales without exhausting computational budgets, we systematically **eliminate unimportant degrees of freedom** and **average out fast modes** into effective potentials, friction, and stochastic thermal noise:

$$\underbrace{\text{Path-Integral MC}}_{\text{Quantum nuclei + electrons}} \longrightarrow \underbrace{\text{Ab Initio MD}}_{\text{DFT electrons, classical nuclei}} \longrightarrow \underbrace{\text{Classical / Semi-Empirical MD}}_{\text{Empirical force fields, }\Delta t \sim 1\text{ fs}} \longrightarrow \underbrace{\text{Langevin Dynamics}}_{\text{Implicit solvent + solute inertia}} \longrightarrow \underbrace{\text{Brownian Dynamics / DPD}}_{\text{Overdamped or coarse-grained mesoscale}} \longrightarrow \underbrace{\text{Metropolis MC}}_{\text{Equilibrium sampling}} \longrightarrow \underbrace{\text{Kinetic MC}}_{\text{Rare-event state-to-state rates}}$$

* **Moving upward** (toward Path-Integral MC and Ab Initio MD) increases **physical accuracy** and first-principles transferability, at the cost of massive CPU expense per femtosecond.
* **Moving downward** (toward Brownian Dynamics, DPD, and Kinetic MC) increases **accessible time and length scales** by orders of magnitude.

### Why Eliminate Explicit Solvent?
In a dilute colloidal suspension, polymer solution, or solvated protein, **over $90\%$ to $99\%$ of the atoms in the simulation box are small solvent molecules** (such as water). In all-atom Molecular Dynamics:
1. Most CPU cycles are spent computing pair forces between water molecules far from the solute of interest.
2. The integration timestep $\Delta t \approx 1\text{ fs}$ is dictated by the fastest bond vibrations and collisions of light solvent atoms, whereas the heavy solute or polymer relaxes over microseconds to milliseconds.

By integrating out the explicit solvent and replacing it with:
* an **effective potential of mean force** $V(\mathbf{R})$ (which includes solvent screening and depletion effects),
* a **viscous drag / friction** term that removes energy, and
* **stochastic thermal kicks** that pump thermal energy back in at temperature $T$,

we reduce the number of tracked degrees of freedom by an order of magnitude *and* increase the allowable timestep $\Delta t$ from femtoseconds to picoseconds or nanoseconds.

---

### Can Monte Carlo Be Used for Dynamics?
Metropolis Monte Carlo was invented solely to sample static equilibrium integrals ($\langle A \rangle_{\text{NVT}}$), so in general **MC trajectories are fictitious** and have no physical clock. However, an MC Markov chain *can* model realistic coarse-grained diffusive relaxation if two strict conditions hold:
1. **Locality of trial moves and conservation laws:** Trial moves must consist exclusively of small, continuous single-particle displacements ($\Delta \mathbf{r}$ small), polymer reptation steps, or nearest-neighbor spin/atom exchanges that respect local conservation laws.
 * In the Ising lattice gas, **Glauber dynamics** flips single spins (non-conserved order parameter, Model A), whereas **Kawasaki dynamics** swaps only adjacent opposite spins ($\sigma_i \leftrightarrow \sigma_j$ on nearest-neighbor bonds), strictly conserving total magnetization/composition (Model B diffusive phase separation) and establishing **local thermodynamic equilibrium**.
 * Non-local moves—such as **Wolff/Swendsen–Wang cluster flips**, **umbrella sampling**, or **global polymer pivot moves**—accelerate equilibrium sampling by bypassing energy barriers, which completely **destroys physical dynamical trajectories**.
2. **High acceptance ratio ($\langle A \rangle \to 1$):** In continuous-space Brownian/Smart MC, rejected trial moves pin the particle in place artificially. Faithful time-dependent diffusion requires small steps where the acceptance probability is close to $100\%$ (typically $\sim 99\%$).


In [3]:
ws.render_section('q01_continuum_ordering')

**Q1 · easy** · 💡

As we move down the hierarchy from Path-Integral Monte Carlo and Ab Initio MD toward Langevin Dynamics, Brownian Dynamics, and Kinetic Monte Carlo, what is the fundamental mathematical strategy used to achieve faster simulations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how fast degrees of freedom (electrons, explicit solvent vibrations) are treated as we coarse-grain.

</div>
</details>

- [ ] **(A)** Eliminating unimportant variables (such as explicit solvent coordinates) and averaging out fast degrees of freedom into effective potentials, friction, and stochastic noise.
- [ ] **(B)** Replacing canonical Boltzmann statistics with microcanonical deterministic trajectories at larger integration timesteps without changing the degrees of freedom.
- [ ] **(C)** Increasing the mass of electrons and solvent atoms to equal the solute mass while keeping all explicit quantum wavefunctions.
- [ ] **(D)** Truncating all interatomic forces at the nearest-neighbor shell while integrating Newton's equations with a 0.1 fs timestep.

In [4]:
ws.render_section('q02_solvent_elimination_motivation')

**Q2 · easy** · 💡

Why does replacing explicit water molecules with an implicit stochastic bath in a biomolecular or colloidal simulation typically reduce computational cost by more than $90\%$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about both the number of pair force evaluations per step and the maximum stable integration timestep.

</div>
</details>

- [ ] **(A)** Water molecules interact only via three-body quantum exchange potentials that scale as O(N^4) in classical MD.
- [ ] **(B)** Explicit water molecules account for $> 90\%$ of the atoms (dominating pair-force evaluations) and their fast O–H vibrations/collisions force tiny $\sim 1\text{ fs}$ timesteps.
- [ ] **(C)** Implicit solvent eliminates the need to satisfy detailed balance or conserve probability.
- [ ] **(D)** Explicit solvent prevents the use of periodic boundary conditions and Verlet neighbor lists.

In [5]:
ws.render_section('q03_kawasaki_vs_nonlocal_mc')

**Q3 · medium** · 💡

Under what condition can a Monte Carlo simulation of a binary alloy or Ising lattice gas be interpreted as a physically realistic model of diffusive time evolution?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare nearest-neighbor Kawasaki exchanges against non-local cluster or global swap moves.

</div>
</details>

- [ ] **(A)** Whenever the Metropolis acceptance criterion is satisfied, regardless of whether atoms swap across the entire simulation box.
- [ ] **(B)** When global Wolff cluster flips are used so that critical slowing down is eliminated near the critical temperature.
- [ ] **(C)** When trial moves are strictly local and respect conservation laws (such as Kawasaki nearest-neighbor spin/atom exchanges), maintaining local thermodynamic equilibrium.
- [ ] **(D)** Only when the acceptance ratio is tuned below 10% so that particles rarely move.

In [6]:
ws.render_section('q04_mc_methods_realistic_dynamics')

**Q4 · medium** · 💡

Which of the following Monte Carlo move schemes produce trajectories that can be mapped onto physically realistic dynamics? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look for local continuous displacements, polymer chain reptation along its contour, nearest-neighbor Kawasaki exchanges, and rate-based Kinetic MC.

</div>
</details>

- [ ] **(A)** Small-step Brownian / Smart Monte Carlo with acceptance ratio near 99%
- [ ] **(B)** Polymer reptation MC (slithering-snake moves along the entangled chain tube)
- [ ] **(C)** Kawasaki nearest-neighbor spin-exchange MC in a conserved binary alloy
- [ ] **(D)** Swendsen–Wang / Wolff non-local cluster spin flips
- [ ] **(E)** Kinetic Monte Carlo (KMC) using transition-state barrier rates

In [7]:
ws.render_section('q05_solvent_cpu_fraction_numeric')

**Q5 · easy** · 💡

A colloidal simulation box contains $N_{\text{solute}} = 400$ heavy polymer beads and $N_{\text{solvent}} = 9{,}600$ explicit solvent atoms ($N_{\text{total}} = 10{,}000$). If short-range neighbor-list force evaluations scale linearly with the total number of particles $N$, what percentage of the per-step force-evaluation CPU time is eliminated when the explicit solvent is replaced by an implicit Brownian bath (assuming the per-particle cost for the remaining $400$ beads is unchanged)? Enter your answer as a percentage between 0 and 100.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the fraction of total particle count represented by the solvent particles and express as a percentage.

</div>
</details>

_Numeric answer required._

In [8]:
ws.render_section('sec_smoluchowski_and_moments')

### Local Markovian Evolution and the Master Equation
Suppose we track the continuous coarse-grained coordinates $\mathbf{R}(t)$ of heavy solute particles after eliminating the fast solvent. Over timescales longer than the solvent collision time, the trajectory is **Markovian** (memoryless) and **local** (continuous paths built from infinitesimal steps).

Let $\rho(\mathbf{R}, \mathbf{R}'; t)$ denote the **Green's function** (transition probability density) of finding the system at configuration $\mathbf{R}$ at time $t$, given that it started at $\mathbf{R}'$ at $t = 0$:

$$\frac{\partial \rho(\mathbf{R}, \mathbf{R}'; t)}{\partial t} = \hat{O}\rho(\mathbf{R}, \mathbf{R}'; t), \qquad \rho(\mathbf{R}, \mathbf{R}'; 0) = \delta(\mathbf{R} - \mathbf{R}')$$

By the Central Limit Theorem, summing many infinitesimal local random steps leaves only two surviving cumulants at order $\Delta t$: the **mean drift** (first spatial derivative) and the **covariance / diffusion** (second spatial derivative). Furthermore, total probability $\int d\mathbf{R}\,\rho(\mathbf{R}, \mathbf{R}'; t) = 1$ must be conserved ($\partial_t \rho = -\nabla\cdot\mathbf{J}$).

### The Smoluchowski Advection-Diffusion Equation
Requiring probability conservation and the canonical Boltzmann distribution as the unique stationary equilibrium yields the **Smoluchowski equation**:

$$\frac{\partial \rho(\mathbf{R}, t)}{\partial t} = \nabla \cdot \left[ \underline{\mathbf{D}}(\mathbf{R}) \left( \nabla \rho(\mathbf{R}, t) - \beta \mathbf{F}(\mathbf{R}) \rho(\mathbf{R}, t) \right) \right]$$

where $\underline{\mathbf{D}}(\mathbf{R})$ is the symmetric positive-definite diffusion tensor, $\beta = 1/(k_B T)$, and $\mathbf{F}(\mathbf{R}) = -\nabla V(\mathbf{R})$ is the conservative force (gradient of the potential of mean force).

#### Proof of Canonical Boltzmann Stationarity
At steady state ($\partial_t \rho_{\text{eq}} = 0$) with zero probability flux ($\mathbf{J} = \mathbf{0}$):

$$\nabla \rho_{\text{eq}}(\mathbf{R}) - \beta \mathbf{F}(\mathbf{R}) \rho_{\text{eq}}(\mathbf{R}) = \mathbf{0} \implies \frac{\nabla \rho_{\text{eq}}}{\rho_{\text{eq}}} = \nabla \ln \rho_{\text{eq}} = -\beta \nabla V(\mathbf{R})$$

Integrating both sides immediately gives the **canonical Boltzmann distribution**:

$$\lim_{t\to\infty} \rho(\mathbf{R}, \mathbf{R}'; t) = \rho_{\text{eq}}(\mathbf{R}) = \frac{e^{-\beta V(\mathbf{R})}}{Z}$$

---

### Short-Time Moment Expansion & The Spurious Drift Correction $\nabla\cdot\underline{\mathbf{D}}$
To construct a stochastic simulation step from $\mathbf{R}'$ at $t=0$, we compute the instantaneous rate of change of the mean position $\langle \mathbf{R} \rangle$ using integration by parts (assuming boundary terms vanish):

$$\left\langle \frac{d\mathbf{R}}{dt} \right\rangle = \int d\mathbf{R}\, \mathbf{R} \frac{\partial \rho}{\partial t} = \int d\mathbf{R}\, \mathbf{R}\, \nabla \cdot \left[ \underline{\mathbf{D}}(\mathbf{R}) (\nabla \rho - \beta \mathbf{F} \rho) \right]$$

Integrating by parts once transfers $\nabla$ onto $\mathbf{R}$ (giving a minus sign):

$$\left\langle \frac{d\mathbf{R}}{dt} \right\rangle = -\int d\mathbf{R}\, \underline{\mathbf{D}}(\mathbf{R}) \left( \nabla \rho(\mathbf{R}, t) - \beta \mathbf{F}(\mathbf{R}) \rho(\mathbf{R}, t) \right)$$

Integrating the $\underline{\mathbf{D}}(\mathbf{R})\nabla\rho$ term by parts a *second* time transfers $\nabla$ onto $\underline{\mathbf{D}}(\mathbf{R})$:

$$\left\langle \frac{d\mathbf{R}}{dt} \right\rangle = \int d\mathbf{R}\, \left[ \nabla \cdot \underline{\mathbf{D}}(\mathbf{R}) + \beta \underline{\mathbf{D}}(\mathbf{R}) \mathbf{F}(\mathbf{R}) \right] \rho(\mathbf{R}, \mathbf{R}'; t)$$

Evaluating at $t \to 0^+$ where $\rho(\mathbf{R}, \mathbf{R}'; 0) = \delta(\mathbf{R} - \mathbf{R}')$ yields the exact short-time **first and second moments**:

$$\left.\left\langle \frac{d\mathbf{R}}{dt} \right\rangle\right|_{t=0} = \nabla \cdot \underline{\mathbf{D}}(\mathbf{R}') + \beta \underline{\mathbf{D}}(\mathbf{R}') \mathbf{F}(\mathbf{R}'), \qquad \left.\frac{d\langle (\mathbf{R} - \mathbf{R}')(\mathbf{R} - \mathbf{R}') \rangle}{dt}\right|_{t=0} = 2\underline{\mathbf{D}}(\mathbf{R}')$$

> **Critical Insight:** Whenever the diffusivity $\underline{\mathbf{D}}(\mathbf{R})$ varies with position (e.g., near a wall, across a membrane, or with hydrodynamic Oseen coupling), the **divergence drift term $\nabla \cdot \underline{\mathbf{D}}(\mathbf{R})$** is mandatory! Omitting $\nabla \cdot \underline{\mathbf{D}}$ violates detailed balance and causes particles to artificially accumulate in low-diffusivity regions.


In [9]:
ws.render_section('q06_smoluchowski_operator_form')

**Q6 · easy** · 💡

Why does the master equation operator $\hat{O}$ for a continuous, local Markovian walk in configuration space contain only first-order (drift) and second-order (diffusion) spatial derivatives?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Invoke the Central Limit Theorem for the sum of many tiny random steps.

</div>
</details>

- [ ] **(A)** Third- and fourth-order derivatives only appear when the temperature is negative.
- [ ] **(B)** Higher spatial derivatives are forbidden by Newton's third law of action and reaction.
- [ ] **(C)** The canonical Boltzmann distribution is a quadratic polynomial in position.
- [ ] **(D)** By the Central Limit Theorem, when a continuous walk is built from many infinitesimal steps, only the first moment (mean drift) and second cumulant (variance/diffusion) survive in the continuum limit.

In [10]:
ws.render_section('q07_smoluchowski_zero_flux_boltzmann')

**Q7 · medium** · 💡

In the Smoluchowski equation $\partial_t \rho = \nabla \cdot [\underline{\mathbf{D}}(\mathbf{R})(\nabla \rho - \beta \mathbf{F}\rho)]$, what relation between the diffusive flux and drift flux guarantees that the stationary state $\partial_t \rho = 0$ is the canonical Boltzmann distribution $\rho_{\text{eq}}(\mathbf{R}) = Z^{-1}e^{-\beta V(\mathbf{R})}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the governing physical expression and substitute the specified parameters.

</div>
</details>

- [ ] **(A)** $\nabla\rho = \beta \mathbf{F}\rho = -\beta(\nabla V)\rho$, so the entropic diffusive flux $-\mathbf{D}\nabla\rho$ exactly balances the force-driven drift flux $+\beta\mathbf{D}\mathbf{F}\rho$ everywhere.
- [ ] **(B)** $\nabla \cdot \mathbf{D}(\mathbf{R}) = 0$ and $\mathbf{F}(\mathbf{R}) = 0$ everywhere in the simulation box.
- [ ] **(C)** $\nabla\rho = -\beta \mathbf{F}\rho$, so diffusion and drift push probability in the same direction.
- [ ] **(D)** $\mathbf{D}(\mathbf{R})$ must be proportional to $\exp(+\beta V(\mathbf{R}))$ while $\rho$ remains spatially uniform.

In [11]:
ws.render_section('q08_spurious_drift_divergence')

**Q8 · medium** · 💡

When the diffusion tensor $\underline{\mathbf{D}}(\mathbf{R})$ depends on particle positions $\mathbf{R}$, what is the exact instantaneous mean drift velocity $\langle d\mathbf{R}/dt \rangle|_{t=0}$ obtained from the moment expansion of the Smoluchowski equation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the second integration by parts of $-\int d\mathbf{R}\, \underline{\mathbf{D}}(\mathbf{R})\nabla\rho$.

</div>
</details>

- [ ] **(A)** $\beta \mathbf{D}(\mathbf{R})\mathbf{F}(\mathbf{R})$ only
- [ ] **(B)** $\nabla \cdot \mathbf{D}(\mathbf{R}) + \beta \mathbf{D}(\mathbf{R})\mathbf{F}(\mathbf{R})$
- [ ] **(C)** $-\nabla \cdot \mathbf{D}(\mathbf{R}) + \beta \mathbf{D}(\mathbf{R})\mathbf{F}(\mathbf{R})$
- [ ] **(D)** $\mathbf{D}(\mathbf{R})\nabla \cdot \mathbf{F}(\mathbf{R})$

In [12]:
ws.render_section('q09_position_dependent_drift_numeric')

**Q9 · medium** · 💡

A colloidal particle diffuses in 1D near a wall with a position-dependent diffusivity $D(x) = D_0 (1 + 0.5 x^2)$ where $D_0 = 2.0\text{ nm}^2/\text{ns}$ and $x$ is in $\text{nm}$. At position $x = 3.0\text{ nm}$, the conservative force is $F(x) = -0.4\,k_B T/\text{nm}$ (so $\beta F(3.0) = -0.4\text{ nm}^{-1}$). Calculate the exact Smoluchowski mean drift velocity $\langle dx/dt \rangle|_{t=0} = \frac{dD}{dx} + \beta D(x) F(x)$ at $x = 3.0\text{ nm}$ in $\text{nm/ns}$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate the diffusion coefficient with respect to $x$ and add the force-driven drift velocity.

</div>
</details>

_Numeric answer required._

In [13]:
ws.render_section('q10_variance_growth_rate_numeric')

**Q10 · easy** · 💡

In 3D isotropic Brownian dynamics with scalar diffusivity $D = 1.25\text{ nm}^2/\text{ns}$, what is the short-time mean-square displacement $\langle |\mathbf{r}(\Delta t) - \mathbf{r}(0)|^2 \rangle$ (in $\text{nm}^2$) of the stochastic displacement step over a time interval $\Delta t = 0.40\text{ ns}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Sum the diffusion variance contributions across all three spatial dimensions.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('sec_trotter_and_smart_mc')

### Trotter's Formula (1959)
The Smoluchowski master equation has the formal operator solution $\hat{\rho}(t) = e^{(\hat{A} + \hat{B})t}\hat{\rho}(0)$, where $\hat{A} = D\nabla^2$ is the **diffusion operator** and $\hat{B} = -\beta D \nabla \cdot (\mathbf{F}\,\cdot)$ is the **drift operator**. Because $[\hat{A}, \hat{B}] \ne 0$, $e^{(\hat{A}+\hat{B})t} \ne e^{\hat{A}t}e^{\hat{B}t}$ for large $t$. However, **Trotter's theorem** guarantees that splitting the evolution into $n$ small steps of size $\tau = t/n$ is exact as $n \to \infty$:

$$e^{(\hat{A} + \hat{B})t} = \lim_{n\to\infty} \left[ e^{\frac{t}{n}\hat{A}} e^{\frac{t}{n}\hat{B}} \right]^n, \qquad e^{\tau(\hat{A}+\hat{B})} = e^{\tau\hat{A}}e^{\tau\hat{B}} + \mathcal{O}(\tau^2)$$

Inserting complete sets of position eigenstates $|\mathbf{r}_k\rangle$ between each Trotter factor turns the time evolution into a Markov chain (or path integral) where we alternate independent **drift** and **diffusion** propagators at each small time step $\tau$:

1. **Diffusion Green's Function ($\hat{T} = -D\nabla^2$):** Expanding in plane-wave eigenfunctions $\phi_{\mathbf{k}}(\mathbf{r}) = \Omega^{-1/2}e^{-i\mathbf{k}\cdot\mathbf{r}}$ with eigenvalue $Dk^2$ and converting the sum to a 3D Fourier integral gives the exact Gaussian heat kernel:
 $$\langle \mathbf{r} | e^{-\tau \hat{T}} | \mathbf{r}' \rangle = \frac{1}{(2\pi)^3} \int d^3k\, e^{i\mathbf{k}\cdot(\mathbf{r}'-\mathbf{r}) - \tau D k^2} = \frac{1}{(4\pi D \tau)^{3/2}} \exp\left( -\frac{|\mathbf{r} - \mathbf{r}'|^2}{4D\tau} \right)$$
2. **Drift Green's Function ($\partial_t \rho = -\mathbf{v}_{\text{drift}}\cdot\nabla\rho$):** For a locally constant force $\mathbf{F}(\mathbf{r}')$, the drift operator simply translates the probability packet deterministically by $\Delta\mathbf{r}_{\text{drift}} = \beta D \tau \mathbf{F}(\mathbf{r}')$:
 $$\rho(\mathbf{r}, \tau) = h\left(\mathbf{r} - \beta D \tau \mathbf{F}(\mathbf{r}')\right)$$

---

### Combining Drift & Diffusion: Brownian Update vs. Hybrid Smart Monte Carlo
Convolving the deterministic drift shift with the Gaussian diffusion kernel yields the **short-time Brownian transition probability (displaced Gaussian proposal)**:

$$T(\mathbf{r} \to \mathbf{r}') = \frac{1}{(4\pi D \tau)^{3/2}} \exp\left[ -\frac{\left(\mathbf{r}' - \mathbf{r} - \beta D \tau \mathbf{F}(\mathbf{r})\right)^2}{4 D \tau} \right]$$

which corresponds to the stochastic **Euler–Maruyama Brownian update**:

$$\mathbf{r}(t + \tau) = \mathbf{r}(t) + \beta D \tau \mathbf{F}(\mathbf{r}(t)) + \boldsymbol{\eta}(t), \qquad \langle \boldsymbol{\eta}(t) \rangle = \mathbf{0}, \quad \langle \eta_\alpha(t)\eta_\beta(t) \rangle = 2 D \tau \delta_{\alpha\beta}$$

#### Enforcing Detailed Balance via Smart Monte Carlo (Hybrid Method)
Because Trotter splitting has $\mathcal{O}(\tau^2)$ local error when $\mathbf{F}(\mathbf{r})$ varies with position, pure Brownian dynamics at finite $\tau$ suffers from systematic **time-step bias** in its steady-state distribution. We eliminate this static error completely by using $T(\mathbf{r} \to \mathbf{r}')$ as a **Metropolis trial proposal** (**Smart Monte Carlo**, Rossky–Doll–Friedman 1978) and accepting the step with probability:

$$A(\mathbf{r} \to \mathbf{r}') = \min\left[ 1,\; \frac{T(\mathbf{r}' \to \mathbf{r})\,e^{-\beta V(\mathbf{r}')}}{T(\mathbf{r} \to \mathbf{r}')\,e^{-\beta V(\mathbf{r})}} \right] = \min\left\{ 1,\; \exp\left( -\beta \left[ V(\mathbf{r}') - V(\mathbf{r}) + \frac{\mathbf{F}(\mathbf{r}') + \mathbf{F}(\mathbf{r})}{2} \cdot \left( \mathbf{r}' - \mathbf{r} + \frac{\beta D \tau}{2}(\mathbf{F}(\mathbf{r}') - \mathbf{F}(\mathbf{r})) \right) \right] \right) \right\}$$

* **Linear potential ($V(\mathbf{r})$ linear, $\mathbf{F}(\mathbf{r}') = \mathbf{F}(\mathbf{r})$):** The exponent inside $A(\mathbf{r}\to\mathbf{r}')$ vanishes identically, so **$A(\mathbf{r}\to\mathbf{r}') = 1$ for any step size $\tau$!**
* **General potential:** Enforcing Metropolis rejections guarantees the **exact canonical Boltzmann distribution** for *any* $\tau$, **BUT** to get **correct time-dependent dynamics** we must choose $\tau$ small enough that the acceptance ratio is close to $1$ (typically $\sim 99\%$).


In [15]:
ws.render_section('concept_smoluchowski_trotter')

### Interactive Explorer: Smoluchowski Propagator, Trotter Step Error, and Hybrid Smart MC

Adjust the reduced step size $\tau = D\Delta t$, potential anharmonicity $k_4$, and view mode to inspect the displaced Gaussian propagator $T(x_0 \to x')$ and compare the stationary probability density of unadjusted Brownian Dynamics against Hybrid Smart Monte Carlo.

**Key Takeaway:** Pure Brownian Dynamics exhibits $\mathcal{O}(\tau)$ steady-state broadening in curved potentials when $\tau$ is large. Hybrid Smart MC restores the exact Boltzmann distribution $e^{-\beta V(x)}/Z$ at all $\tau$ via rejections, while keeping $\tau$ small ($\langle A \rangle \approx 99\%$) ensures both exact statics and faithful Smoluchowski dynamics.

In [16]:
ws.render_section('q11_trotter_splitting_purpose')

**Q11 · easy** · 💡

Why do we need Trotter's formula $e^{(\hat{A}+\hat{B})t} = \lim_{n\to\infty}[e^{(t/n)\hat{A}}e^{(t/n)\hat{B}}]^n$ to construct the short-time propagator for Brownian dynamics in an external potential $V(\mathbf{r})$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider whether the diffusion operator $\hat{A} = D\nabla^2$ and the position-dependent drift operator $\hat{B} = -\beta D\nabla\cdot(\mathbf{F}(\mathbf{r})\,\cdot)$ commute.

</div>
</details>

- [ ] **(A)** Because Trotter splitting converts a first-order differential equation in time into a fourth-order Runge–Kutta integrator.
- [ ] **(B)** Because the diffusion operator $D\nabla^2$ has no eigenfunctions in three dimensions.
- [ ] **(C)** Because the diffusion operator $A = D\nabla^2$ and the position-dependent force-drift operator $B$ do not commute ($[A, B] \ne 0$), so their exponentials can only be factored independently over infinitesimal time steps $\tau = t/n$.
- [ ] **(D)** Because the drift operator alone already generates the exact Gaussian heat kernel without needing the Laplacian.

In [17]:
ws.render_section('q12_diffusion_greens_function_eigenbasis')

**Q12 · medium** · 💡

When deriving the 3D free-diffusion Green's function $\langle \mathbf{r} | e^{-\tau \hat{T}} | \mathbf{r}' \rangle$ for $\hat{T} = -D\nabla^2$, we expand in plane-wave eigenfunctions $\phi_{\mathbf{k}}(\mathbf{r}) = \Omega^{-1/2}e^{-i\mathbf{k}\cdot\mathbf{r}}$. What is the eigenvalue of $\hat{T}$ acting on $\phi_{\mathbf{k}}(\mathbf{r})$, and what real-space propagator results after Fourier integration?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply $-D\nabla^2$ to $e^{-i\mathbf{k}\cdot\mathbf{r}}$ and take the Gaussian Fourier transform of $e^{-\tau D k^2}$.

</div>
</details>

- [ ] **(A)** Eigenvalue $D^2 k^4$, yielding a uniform hard-sphere step inside radius $\sqrt{2D\tau}$.
- [ ] **(B)** Eigenvalue $-D k^2$, yielding an oscillatory cosine kernel $\cos\left(\frac{|\mathbf{r} - \mathbf{r}'|}{\sqrt{D\tau}}\right)$.
- [ ] **(C)** Eigenvalue $D|k|$, yielding a Lorentzian Cauchy kernel $\frac{1}{(|\mathbf{r} - \mathbf{r}'|^2 + (D\tau)^2)^2}$.
- [ ] **(D)** Eigenvalue $D k^2$, yielding the Gaussian kernel $(4\pi D\tau)^{-3/2} \exp\left(-\frac{|\mathbf{r} - \mathbf{r}'|^2}{4D\tau}\right)$.

In [18]:
ws.render_section('q13_drift_operator_greens_function')

**Q13 · easy** · 💡

For the 1D pure drift equation $\frac{\partial \rho(x, t)}{\partial t} = -v_d \frac{\partial \rho(x, t)}{\partial x}$ with constant drift velocity $v_d = \beta D F$ and initial profile $\rho(x, 0) = h(x)$, what is the exact time-evolved solution $\rho(x, t)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check how a pure advection operator translates the initial shape $h(x)$.

</div>
</details>

- [ ] **(A)** $\rho(x, t) = h(x - v_d t)$, a rigid translation of the probability distribution in the direction of $F$ without spreading.
- [ ] **(B)** $\rho(x, t) = h(x)\exp(-v_d t)$, an exponential decay in amplitude at fixed position.
- [ ] **(C)** $\rho(x, t) = h(x + v_d t^2)$, a uniformly accelerated wave packet.
- [ ] **(D)** $\rho(x, t) = (4\pi v_d t)^{-1/2}\exp\left(-\frac{x^2}{4 v_d t}\right)$, pure diffusive spreading with zero mean shift.

In [19]:
ws.render_section('q14_smart_mc_linear_potential_acceptance')

**Q14 · medium** · 💡

In Smart Monte Carlo (Hybrid Brownian MC), a trial displacement $\Delta \mathbf{r} = \tau\beta\mathbf{F}(\mathbf{r}) + \boldsymbol{\delta}$ (with $\langle \delta^2 \rangle = 2\tau$, setting $D=1$) is proposed from the displaced Gaussian $T(\mathbf{r}\to\mathbf{r}')$. What is the Metropolis acceptance probability $A(\mathbf{r}\to\mathbf{r}')$ if the potential $V(\mathbf{r}) = V_0 - \mathbf{F}_0 \cdot \mathbf{r}$ is strictly linear (such as a uniform gravitational or electric field on a colloid)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $\mathbf{F}(\mathbf{r}') = \mathbf{F}(\mathbf{r}) = \mathbf{F}_0$ and $V(\mathbf{r}') - V(\mathbf{r}) = -\mathbf{F}_0 \cdot (\mathbf{r}' - \mathbf{r})$ into the Smart MC acceptance exponent.

</div>
</details>

- [ ] **(A)** $A(\mathbf{r} \to \mathbf{r}') = \exp(-\beta F_0^2 \tau / 2)$, decaying exponentially with step size $\tau$.
- [ ] **(B)** $A(\mathbf{r} \to \mathbf{r}') = 1$ identically for every trial move and any step size $\tau$.
- [ ] **(C)** $A(\mathbf{r} \to \mathbf{r}') = 0.50$ because half of the Gaussian steps move uphill against $F_0$.
- [ ] **(D)** $A(\mathbf{r} \to \mathbf{r}') = \min(1, \exp(-\beta[V(\mathbf{r}') - V(\mathbf{r})]))$, identical to unshifted Metropolis MC.

In [20]:
ws.render_section('q15_smart_mc_acceptance_exponent_numeric')

**Q15 · hard** · 💡

Consider a 1D Smart Monte Carlo step with $\beta = 1$ and $\tau = D\Delta t = 0.20$ in a harmonic potential $V(x) = \frac{1}{2}k x^2$ with spring constant $k = 2.0$ (so $F(x) = -2x$). Starting from the minimum $x = 0.0$ (where $V(0.0) = 0.0$ and $F(0.0) = 0.0$), a trial move lands at $x' = 1.0$ (where $V(1.0) = 1.0$ and $F(1.0) = -2.0$). Using the Smart MC acceptance formula:

$$A(x \to x') = \min\left(1,\; \exp\left\{ -\beta\left[ V(x') - V(x) + \frac{F(x') + F(x)}{2}\left( (x' - x) + \frac{\tau\beta}{2}(F(x') - F(x)) \right) \right] \right\}\right)$$

calculate the acceptance probability $A(0.0 \to 1.0)$ (round to 3 decimal places).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the terms in the Smart MC acceptance exponent using the harmonic potential energies and forces at the initial and trial endpoints.

</div>
</details>

_Numeric answer required._

In [21]:
ws.render_section('q16_hybrid_brownian_acceptance_target')

**Q16 · easy** · 💡

In the Hybrid Brownian / Smart Monte Carlo algorithm, enforcing the Metropolis acceptance test guarantees the exact canonical Boltzmann steady state for any step size $\tau$. Why do we still restrict the time step $\tau$ so that the acceptance ratio is very close to $1$ (typically $\sim 99\%$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Distinguish between static equilibrium averages and time-dependent dynamical trajectories.

</div>
</details>

- [ ] **(A)** Because random number generators cannot sample Gaussian distributions when $\tau > 0.01$.
- [ ] **(B)** Because the canonical partition function Z diverges whenever the acceptance rate drops below 95%.
- [ ] **(C)** Because rejected moves artificially halt particle motion and distort time correlation functions; high acceptance (~99%) is required to keep the dynamical trajectory faithful to the Smoluchowski equation.
- [ ] **(D)** Because detailed balance is violated whenever a single trial move is rejected.

In [22]:
ws.render_section('sec_hydrodynamic_interactions_oseen')

### Why Simple Brownian Dynamics Misses Long-Time Hydrodynamic Correlations
In simple (free-draining) Brownian dynamics, each particle $i$ is assigned an independent scalar diffusivity $D_0 = \frac{k_B T}{6\pi\eta r_0}$ (Stokes–Einstein relation for a sphere of radius $r_0$ in a solvent of viscosity $\eta$) and uncorrelated thermal noise. However, in a real fluid, **when particle $i$ moves, it drags the surrounding solvent, creating a long-range velocity field that pushes particle $j$**.

### The Oseen Hydrodynamic Mobility Tensor
Within linear Stokes hydrodynamics (low Reynolds number), the velocity perturbation at distance $r_{ij}$ from a point force in an incompressible fluid of viscosity $\eta$ decays as **$1/r_{ij}$**. Perturbation theory yields the **Oseen diffusion tensor** $\underline{\mathbf{D}}_{ij}(\mathbf{r}_{ij})$:

$$\underline{\mathbf{D}}_{ij} = \frac{k_B T}{\pi \eta} \begin{cases} \dfrac{1}{6 r_0}\mathbf{I} &: i = j \\[10pt] \dfrac{1}{8 r_{ij}} \left( \mathbf{I} + \dfrac{\mathbf{r}_{ij}\mathbf{r}_{ij}}{r_{ij}^2} \right) &: i \ne j \end{cases}$$

Notice the directional anisotropy for a pair separated along unit vector $\hat{\mathbf{r}}_{ij} = \mathbf{r}_{ij}/r_{ij}$:
* **Longitudinal coupling** (motion parallel to $\hat{\mathbf{r}}_{ij}$, where $\hat{\mathbf{r}}_{ij}\cdot(\mathbf{I} + \hat{\mathbf{r}}_{ij}\hat{\mathbf{r}}_{ij})\cdot\hat{\mathbf{r}}_{ij} = 2$):
 $$D_{\parallel}(r_{ij}) = \frac{k_B T}{4\pi\eta r_{ij}} = \frac{3 r_0}{2 r_{ij}} D_0$$
* **Transverse coupling** (motion perpendicular to $\hat{\mathbf{r}}_{ij}$, where $\hat{\mathbf{e}}_\perp\cdot(\mathbf{I} + \hat{\mathbf{r}}_{ij}\hat{\mathbf{r}}_{ij})\cdot\hat{\mathbf{e}}_\perp = 1$):
 $$D_{\perp}(r_{ij}) = \frac{k_B T}{8\pi\eta r_{ij}} = \frac{3 r_0}{4 r_{ij}} D_0 = \frac{1}{2} D_{\parallel}(r_{ij})$$

---

### Generating Correlated Thermal Noise via Cholesky Decomposition
Because the $3N \times 3N$ diffusion matrix $\underline{\mathbf{D}}(\mathbf{R})$ couples different particles ($i \ne j$), the Fluctuation–Dissipation theorem requires that the **stochastic Brownian displacements $\boldsymbol{\eta}_i$ and $\boldsymbol{\eta}_j$ must be correlated**:

$$\langle \boldsymbol{\eta}_i \boldsymbol{\eta}_j^T \rangle = 2 \underline{\mathbf{D}}_{ij}(\mathbf{R}) \Delta t$$

To sample a $3N$-dimensional correlated Gaussian vector $\boldsymbol{\eta}$ from a vector $\boldsymbol{\xi}$ of independent standard normal variates ($\langle \xi_\mu \xi_\nu \rangle = \delta_{\mu\nu}$), we factorize the $3N \times 3N$ matrix $\underline{\mathbf{D}}$ via **Cholesky decomposition**:

$$\underline{\mathbf{D}}(\mathbf{R}) = \mathbf{S}\mathbf{S}^T \implies \boldsymbol{\eta} = \sqrt{2\Delta t}\,\mathbf{S}\boldsymbol{\xi}$$

**Important limitations of Oseen Brownian Dynamics:**
1. **$\mathcal{O}(N^3)$ scaling bottleneck:** Because $\underline{\mathbf{D}}_{ij} \propto 1/r_{ij}$ is dense and depends on instantaneous positions $\mathbf{R}(t)$, Cholesky factorization costs $\mathcal{O}((3N)^3) = \mathcal{O}(N^3)$ operations every time $\underline{\mathbf{D}}$ is updated.
2. **Linear approximation:** The Oseen tensor is only a far-field linear approximation ($r_{ij} \gg r_0$) and can fail to be positive-definite when particles overlap unless regularized (e.g., via the Rotne–Prager–Yamakawa tensor).
3. **Transport coefficients are inputs, not outputs:** Unlike MD, both $D_0$ and the solvent viscosity $\eta$ must be supplied as **external inputs** to a Brownian simulation rather than emerging automatically from microscopic collisions.


In [23]:
ws.render_section('concept_oseen_coupling')

### Interactive Explorer: Hydrodynamic Oseen Tensor & Directional Coupling

Explore how the solvent-mediated pair diffusivity $D_{ij}/D_0$ decays as $1/r_{ij}$ and depends on the angle $\theta$ between the displacement direction and the interparticle axis $\hat{\mathbf{r}}_{ij}$.

**Key Takeaway:** Longitudinal hydrodynamic coupling ($\theta = 0^\circ$) is exactly twice as strong as transverse coupling ($\theta = 90^\circ$). Because $1/r_{ij}$ is long-ranged, all $N$ particles are hydrodynamically coupled through the dense $3N \times 3N$ diffusion matrix.

In [24]:
ws.render_section('q17_oseen_tensor_decay_and_ratio')

**Q17 · easy** · 💡

According to the Oseen tensor $\underline{\mathbf{D}}_{ij}(\mathbf{r}_{ij}) = \frac{k_B T}{8\pi\eta r_{ij}}(\mathbf{I} + \hat{\mathbf{r}}_{ij}\hat{\mathbf{r}}_{ij})$ for $i \ne j$, how does the hydrodynamic coupling decay with interparticle distance $r_{ij}$, and what is the ratio of longitudinal ($D_\parallel$) to transverse ($D_\perp$) pair diffusivity?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $\hat{\mathbf{u}}\cdot(\mathbf{I} + \hat{\mathbf{r}}_{ij}\hat{\mathbf{r}}_{ij})\cdot\hat{\mathbf{u}}$ for $\hat{\mathbf{u}} \parallel \hat{\mathbf{r}}_{ij}$ and $\hat{\mathbf{u}} \perp \hat{\mathbf{r}}_{ij}$.

</div>
</details>

- [ ] **(A)** It decays as $1/r_{ij}^6$ like dispersion forces, and $D_\parallel / D_\perp = 4$.
- [ ] **(B)** It decays as $1/r_{ij}^3$, and longitudinal coupling is equal to transverse coupling ($D_\parallel / D_\perp = 1$).
- [ ] **(C)** It decays exponentially as $\exp(-\kappa r_{ij})$, and $D_\parallel / D_\perp = 1/2$.
- [ ] **(D)** It decays as $1/r_{ij}$, and longitudinal coupling is twice as large as transverse coupling ($D_\parallel / D_\perp = 2$).

In [25]:
ws.render_section('q18_oseen_longitudinal_value_numeric')

**Q18 · medium** · 💡

Two identical spherical colloids of hydrodynamic radius $r_0 = 50\text{ nm}$ have single-particle Stokes–Einstein diffusivity $D_0 = 4.0\text{ }\mu\text{m}^2/\text{s}$. When their center-to-center separation is $r_{12} = 300\text{ nm}$, what is the longitudinal Oseen cross-diffusivity $D_{12,\parallel}$ in $\mu\text{m}^2/\text{s}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the longitudinal Oseen tensor prefactor to the single-particle Stokes-Einstein diffusivity via the ratio of hydrodynamic radius to interparticle distance.

</div>
</details>

_Numeric answer required._

In [26]:
ws.render_section('q19_cholesky_sampling_complexity')

**Q19 · medium** · 💡

Why does incorporating Oseen hydrodynamic interactions into Brownian dynamics increase the per-step computational complexity from $\mathcal{O}(N)$ (or $\mathcal{O}(N^2)$) to $\mathcal{O}(N^3)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how correlated thermal noise $\boldsymbol{\eta} = \sqrt{2\Delta t}\,\mathbf{S}\boldsymbol{\xi}$ is generated from the dense $3N \times 3N$ diffusion matrix $\underline{\mathbf{D}}$.

</div>
</details>

- [ ] **(A)** Because the thermal noise on all $N$ particles is correlated through the dense $3N \times 3N$ Oseen matrix $\mathbf{D}$, requiring an $\mathcal{O}(N^3)$ Cholesky decomposition $\mathbf{D} = \mathbf{S}\mathbf{S}^T$ to sample the noise.
- [ ] **(B)** Because the Oseen tensor requires summing over all triplets (i, j, k) of solvent molecules.
- [ ] **(C)** Because the 1D drift operator must be inverted using a 3D Fast Fourier Transform of size N × N × N.
- [ ] **(D)** Because the Metropolis acceptance rate drops to $\mathcal{O}(N^{-3})$ whenever viscosity $\eta$ is nonzero.

In [27]:
ws.render_section('q20_oseen_divergence_free_property')

**Q20 · hard** · 💡

In the Smoluchowski mean drift formula $\langle d\mathbf{R}_i/dt \rangle = \sum_j \nabla_{\mathbf{r}_j} \cdot \underline{\mathbf{D}}_{ij} + \beta \sum_j \underline{\mathbf{D}}_{ij}\mathbf{F}_j$, what happens to the divergence drift term $\nabla_{\mathbf{r}_j} \cdot \underline{\mathbf{D}}_{ij}(\mathbf{r}_{ij})$ for the 3D Oseen tensor $\underline{\mathbf{D}}_{ij} \propto \frac{1}{r_{ij}}(\mathbf{I} + \hat{\mathbf{r}}_{ij}\hat{\mathbf{r}}_{ij})$ ($i \ne j$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall that the Oseen tensor is the Green's function of an *incompressible* fluid ($\nabla \cdot \mathbf{v} = 0$).

</div>
</details>

- [ ] **(A)** It diverges as $1/r_{ij}^2$ and dominates over all conservative forces at large distances.
- [ ] **(B)** It vanishes identically ($\nabla_{\mathbf{r}_j} \cdot \mathbf{D}_{ij} = 0$ for $i \ne j$) because the Oseen velocity field describes an incompressible fluid ($\nabla \cdot \mathbf{v} = 0$).
- [ ] **(C)** It equals a constant vector pointing toward the center of mass of the simulation box.
- [ ] **(D)** It is proportional to the particle mass $m_i$ divided by the timestep $\Delta t$.

In [28]:
ws.render_section('q21_brownian_input_vs_output')

**Q21 · easy** · 💡

In an all-atom Molecular Dynamics simulation, the diffusion coefficient $D$ and shear viscosity $\eta$ are emergent *outputs* measured via Green–Kubo or Einstein relations. How are $D_0$ and $\eta$ treated in a Brownian Dynamics simulation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the last bullet on the 'Hydrodynamical effects' lecture slide.

</div>
</details>

- [ ] **(A)** $D_0$ and $\eta$ are always set to zero in Brownian dynamics.
- [ ] **(B)** $D_0$ and $\eta$ are computed automatically from the electronic wavefunction at each Brownian step.
- [ ] **(C)** $D_0$ and $\eta$ are phenomenological inputs supplied to the simulation, not emergent outputs.
- [ ] **(D)** $D_0$ is an output of the Metropolis rejection step, whereas $\eta$ is fixed at infinity.

In [29]:
ws.render_section('sec_langevin_dynamics_and_fdt')

### Retaining Particle Momentum: The Langevin Equation
Brownian dynamics assumes the **overdamped limit** where particle inertia $m\ddot{\mathbf{r}}$ is completely negligible. If we want more realistic short-time dynamics—or need to thermostat an atomic/molecular system while retaining particle velocities—we keep the momentum $\mathbf{p}_i = m\mathbf{v}_i$ of the heavy particle and model the eliminated solvent via a **viscous friction $-\zeta \mathbf{p}_i$** plus a **stochastic solvent force $\mathbf{p}_{\text{solvent}}(t)$** (often written $\boldsymbol{\eta}_i(t)$):

$$\frac{d\mathbf{r}}{dt} = \frac{\mathbf{p}}{m}, \qquad \frac{d\mathbf{p}}{dt} = \mathbf{F}(\mathbf{r}) - \zeta \mathbf{p} + \mathbf{p}_{\text{solvent}}(t)$$

where $\zeta$ (with units of $\text{time}^{-1}$) is the friction coefficient (collision frequency).

### The Fluctuation–Dissipation Theorem (Detailed Balance in Phase Space)
Friction $-\zeta\mathbf{p}$ continuously drains kinetic energy from the particle, whereas the random force $\mathbf{p}_{\text{solvent}}(t)$ continuously injects kinetic energy. To maintain canonical thermal equilibrium at temperature $T$ ($\langle \frac{p_\alpha^2}{2m} \rangle = \frac{1}{2}k_B T$) and satisfy detailed balance, the random force variance **must** be locked to the friction $\zeta$ via the **Fluctuation–Dissipation relation**:

$$\langle \mathbf{p}_{\text{solvent}}(t) \rangle = \mathbf{0}, \qquad \left\langle p_{\text{solvent},\alpha}(t)\, p_{\text{solvent},\beta}(t') \right\rangle = 2 m k_B T \zeta \, \delta_{\alpha\beta}\,\delta(t - t')$$

If the solvent has a finite relaxation time (non-Markovian bath), the delta function is generalized to a memory kernel $\langle p_{\text{solvent}}(t)p_{\text{solvent}}(t') \rangle = C(t - t')$ in the **Generalized Langevin Equation (GLE)**.

---

### Asymptotic Regimes of the Langevin Equation
The friction coefficient $\zeta$ defines the **momentum relaxation time** $\tau_p = \zeta^{-1}$:
1. **Zero-Friction Limit ($\zeta \to 0$):** Both friction and random noise vanish ($\sigma_{\text{noise}} \propto \sqrt{\zeta} \to 0$), and **deterministic Newtonian Molecular Dynamics (NVE)** is exactly recovered.
2. **Short-Time Inertial / Ballistic Regime ($t \ll \zeta^{-1}$):** Before solvent collisions damp its initial velocity, a particle moves ballistically:
 $$C_{vv}(t) = \frac{k_B T}{m} e^{-\zeta t} \approx \frac{k_B T}{m}(1 - \zeta t), \qquad \langle \Delta \mathbf{r}^2(t) \rangle \approx \frac{3 k_B T}{m} t^2 \propto t^2$$
3. **Long-Time / High-Friction Overdamped Limit ($t \gg \zeta^{-1}$ or $\zeta \to \infty$):** Momentum relaxes quasi-instantaneously ($\frac{d\mathbf{p}}{dt} \approx \mathbf{0}$), and the Langevin equation reduces to the **Smoluchowski / Brownian Dynamics** equation with Einstein diffusivity:
 $$D = \int_0^\infty C_{vv}(t)\,dt = \frac{k_B T}{m} \int_0^\infty e^{-\zeta t}\,dt = \frac{k_B T}{m \zeta}, \qquad \langle \Delta \mathbf{r}^2(t) \rangle \simeq 6 D t = \frac{6 k_B T}{m\zeta} t$$

*(Note on Velocity Autocorrelation Functions in dense liquids, Allen & Tildesley Ch. 6: In an explicit dense liquid near the triple point, $C_{vv}(t)$ exhibits a negative "backscattering" cage rebound at $t \sim 0.2\text{ ps}$ before decaying, and in anisotropic molecules like $\text{CS}_2$, parallel and perpendicular diffusive components $D_\parallel \ne D_\perp$ split across temperatures.)*


In [30]:
ws.render_section('concept_langevin_regimes')

### Interactive Explorer: Langevin Friction Regimes (Ballistic vs. Overdamped Brownian)

Vary the Langevin friction coefficient $\zeta$ and switch between the log-log Mean Square Displacement $\langle \Delta \mathbf{r}^2(t) \rangle$ and the Velocity Autocorrelation Function $C_{vv}(t)$ to observe the crossover at $\tau_p = \zeta^{-1}$.

**Key Takeaway:** Langevin dynamics bridges Newtonian MD ($t \ll \zeta^{-1}$, ballistic $\langle \Delta r^2 \rangle \propto t^2$) and Brownian dynamics ($t \gg \zeta^{-1}$, diffusive $\langle \Delta r^2 \rangle = 6Dt$ with $D = k_B T / (m\zeta)$). In a harmonic trap, increasing $\zeta$ above $2\omega_0$ suppresses oscillatory ringing and enters the overdamped Smoluchowski regime.

In [31]:
ws.render_section('q22_langevin_fdt_relation')

**Q22 · easy** · 💡

In the Langevin equation $\frac{d\mathbf{p}}{dt} = \mathbf{F} - \zeta\mathbf{p} + \mathbf{p}_{\text{solvent}}(t)$, what autocorrelation must the stochastic force $\mathbf{p}_{\text{solvent}}(t)$ satisfy per Cartesian component to maintain canonical detailed balance at temperature $T$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the Fluctuation–Dissipation formula on the 'Langevin Equation' lecture slide.

</div>
</details>

- [ ] **(A)** $\langle p_{\text{solvent}}(t) p_{\text{solvent}}(t') \rangle = 0$ for all $t$ and $t'$
- [ ] **(B)** $\langle p_{\text{solvent}}(t) p_{\text{solvent}}(t') \rangle = \frac{k_B T}{\zeta} \delta(t - t')$
- [ ] **(C)** $\langle p_{\text{solvent}}(t) p_{\text{solvent}}(t') \rangle = 2 m^2 \zeta^2 \delta(t - t')$ independent of temperature $T$
- [ ] **(D)** $\langle p_{\text{solvent}}(t) p_{\text{solvent}}(t') \rangle = 2 m k_B T \zeta \delta(t - t')$

In [32]:
ws.render_section('q23_langevin_asymptotic_limits')

**Q23 · easy** · 💡

What happens to the Langevin equation in the two opposite asymptotic limits of the friction coefficient: (i) $\zeta \to 0$ and (ii) $\zeta \to \infty$ (overdamped limit)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Inspect how the inertial term $dp/dt$ behaves in both limits.

</div>
</details>

- [ ] **(A)** As $\zeta \to 0$, deterministic Newtonian MD is recovered; as $\zeta \to \infty$, the overdamped Smoluchowski (Brownian dynamics) equation is recovered.
- [ ] **(B)** As $\zeta \to 0$, the Smoluchowski equation is recovered; as $\zeta \to \infty$, energy-conserving NVE MD is recovered.
- [ ] **(C)** In both limits $\zeta \to 0$ and $\zeta \to \infty$, particles freeze motionless at their initial coordinates.
- [ ] **(D)** As $\zeta \to 0$, Kinetic Monte Carlo is recovered; as $\zeta \to \infty$, Path-Integral MC is recovered.

In [33]:
ws.render_section('q24_langevin_einstein_diffusivity_numeric')

**Q24 · medium** · 💡

In a Langevin simulation run in reduced units where particle mass $m = 2.0$, thermal energy $k_B T = 1.5$, and friction coefficient $\zeta = 0.50\text{ }\tau_0^{-1}$, what is the asymptotic long-time self-diffusion coefficient $D$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the self-diffusion coefficient to the thermal energy, particle mass, and friction coefficient via the Einstein relation.

</div>
</details>

_Numeric answer required._

In [34]:
ws.render_section('q25_langevin_discrete_noise_std_numeric')

**Q25 · medium** · 💡

When integrating the 1D Langevin momentum equation $\Delta p = F\Delta t - \zeta p \Delta t + \Delta W_p$ over a small timestep $\Delta t = 0.01$, the discrete momentum kick $\Delta W_p = \int_t^{t+\Delta t} p_{\text{solvent}}(t')\,dt'$ has variance $\langle (\Delta W_p)^2 \rangle = 2 m k_B T \zeta \Delta t$. For $m = 1.0$, $k_B T = 2.0$, and $\zeta = 4.0$, what is the standard deviation $\sigma_{\Delta W_p} = \sqrt{2 m k_B T \zeta \Delta t}$ of the Gaussian momentum kick per step?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Take the square root of the discrete momentum kick variance to obtain the standard deviation.

</div>
</details>

_Numeric answer required._

In [35]:
ws.render_section('q26_vacf_dense_liquid_cage_effect')

**Q26 · medium** · 💡

Compare the velocity autocorrelation function $C_{vv}(t)$ of a free memoryless Langevin particle ($C_{vv}(t) \propto e^{-\zeta t} > 0$) against the MD velocity autocorrelation function of a dense Lennard-Jones liquid near its triple point (Allen & Tildesley Ch. 6). Why does the dense liquid $C_{vv}(t)$ dip below zero ($C_{vv}(t) < 0$) at intermediate times before decaying to zero?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about what happens when an atom in a dense liquid collides with its cage of nearest neighbors.

</div>
</details>

- [ ] **(A)** Violation of energy conservation in the Verlet integrator at low temperatures.
- [ ] **(B)** Cage backscattering: atoms rebound off their surrounding shell of nearest neighbors, reversing their velocity direction on average before escaping.
- [ ] **(C)** The diffusion coefficient $D = \int_0^\infty C_{vv}(t)\,dt$ is strictly negative in the liquid phase.
- [ ] **(D)** Periodic boundary conditions force every atom to reverse direction when crossing x = L/2.

In [36]:
ws.render_section('q27_langevin_msd_ballistic_vs_diffusive')

**Q27 · medium** · 💡

On a log–log plot of the 3D mean-square displacement $\langle \Delta \mathbf{r}^2(t) \rangle$ versus time $t$ for a free Langevin particle, what are the power-law slopes $\alpha$ (where $\langle \Delta \mathbf{r}^2(t) \rangle \propto t^\alpha$) in the short-time inertial regime ($t \ll \zeta^{-1}$) and the long-time Brownian regime ($t \gg \zeta^{-1}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $\langle \Delta \mathbf{r}^2 \rangle \approx (3k_B T/m)t^2$ at short times with $\langle \Delta \mathbf{r}^2 \rangle \approx 6Dt$ at long times.

</div>
</details>

- [ ] **(A)** Slope $\alpha = 1/2$ at all times.
- [ ] **(B)** Slope $\alpha = 1$ at short times transitioning to slope $\alpha = 2$ at long times.
- [ ] **(C)** Slope $\alpha = 2$ at short times (ballistic regime) transitioning at $t \sim \zeta^{-1}$ to slope $\alpha = 1$ at long times (diffusive regime).
- [ ] **(D)** Slope $\alpha = 3$ at short times transitioning to slope $\alpha = 0$ at long times.

In [37]:
ws.render_section('sec_dpd_and_hydrodynamics')

### Why Global Langevin and Andersen Thermostats Destroy Hydrodynamics
In standard Langevin dynamics (or an Andersen stochastic collision thermostat), the friction force $-\zeta \mathbf{p}_i$ and random kick $\boldsymbol{\eta}_i$ act on each particle's **absolute laboratory velocity** $\mathbf{v}_i$ independently:

$$\sum_{i=1}^N \mathbf{F}_i^{\text{Langevin}} = -\zeta \sum_{i=1}^N \mathbf{p}_i + \sum_{i=1}^N \boldsymbol{\eta}_i(t) \ne \mathbf{0}$$

Because **total momentum is not conserved locally**, a Langevin thermostat breaks **Galilean invariance**:
* Any macroscopic flow profile $\mathbf{u}(\mathbf{r}, t)$ or hydrodynamic vortex experiences an unphysical body drag $-\rho\zeta\mathbf{u}$ against the stationary simulation box frame.
* Long-wavelength shear waves and hydrodynamic wakes are screened out beyond the Brinkman length $\ell_B = \sqrt{\eta/(\rho\zeta)}$, turning parabolic Poiseuille channel flow into flat plug flow and suppressing the long-time $t^{-3/2}$ hydrodynamic tail.

---

### Dissipative Particle Dynamics (DPD): Pairwise Momentum Conservation
**Dissipative Particle Dynamics (DPD)** (Hoogerbrugge & Koelman 1992; Español & Warren 1995; Frenkel & Smit p. 465) solves this problem without explicit solvent atoms by making both the **dissipative (friction)** and **random** forces act **pairwise and centrally** between particles $i$ and $j$ within a cutoff radius $r_c$:

$$\mathbf{F}_i = \sum_{j \ne i} \left( \mathbf{F}_{ij}^C + \mathbf{F}_{ij}^D + \mathbf{F}_{ij}^R \right) = \sum_{j \ne i} \left[ f(r_{ij})\hat{\mathbf{r}}_{ij} + \zeta_{ij}\hat{\mathbf{r}}_{ij} \right]$$

where $\mathbf{r}_{ij} = \mathbf{r}_i - \mathbf{r}_j$, $\hat{\mathbf{r}}_{ij} = \mathbf{r}_{ij}/r_{ij}$, and $\mathbf{v}_{ij} = \mathbf{v}_i - \mathbf{v}_j$ is the **relative velocity**:

1. **Conservative Force $\mathbf{F}_{ij}^C = f(r_{ij})\hat{\mathbf{r}}_{ij}$:** Typically a soft, finite repulsive potential cut off at $r_c$ allowing large timesteps.
2. **Pairwise Dissipative Force $\mathbf{F}_{ij}^D$:** Depends only on the relative velocity projected along the line of centers:
 $$\langle \zeta_{ij} \rangle \hat{\mathbf{r}}_{ij} = \mathbf{F}_{ij}^D = -\gamma w^D(r_{ij}) \left( \mathbf{v}_{ij} \cdot \hat{\mathbf{r}}_{ij} \right) \hat{\mathbf{r}}_{ij}$$
3. **Pairwise Random Force $\mathbf{F}_{ij}^R$:** Symmetric Gaussian noise $\theta_{ij} = \theta_{ji}$ applied equally and oppositely along $\hat{\mathbf{r}}_{ij}$:
 $$\mathbf{F}_{ij}^R = \sigma w^R(r_{ij}) \theta_{ij} \Delta t^{-1/2} \hat{\mathbf{r}}_{ij}$$

*(In the compact notation where $\gamma = 1$ and $\omega(r) = w^R(r)$: $\langle \zeta_{ij} \rangle = -\omega^2(r_{ij})(\mathbf{v}_{ij}\cdot\hat{\mathbf{r}}_{ij})$ and $\langle \zeta_{ij}^2 \rangle = 2 k_B T \omega^2(r_{ij})$.)*

#### Why DPD Recovers Macroscopic Navier–Stokes Hydrodynamics
* **Linear Momentum Conservation:** Because $\mathbf{v}_{ij} = -\mathbf{v}_{ji}$, $\hat{\mathbf{r}}_{ij} = -\hat{\mathbf{r}}_{ji}$, and $\theta_{ij} = \theta_{ji}$, every pair interaction satisfies Newton's third law **$\mathbf{F}_{ij} = -\mathbf{F}_{ji}$** identically! Thus $\sum_i \mathbf{F}_i = \mathbf{0}$ in every region of fluid.
* **Angular Momentum Conservation:** Because all three forces act strictly along the central vector $\hat{\mathbf{r}}_{ij}$, $\mathbf{r}_{ij} \times \mathbf{F}_{ij} = \mathbf{0}$, so local angular momentum is also rigorously conserved.
* **DPD Fluctuation–Dissipation Theorem:** As proved by Español & Warren (1995), the Fokker–Planck equation of DPD has the **canonical Boltzmann distribution** $\rho_{\text{eq}} \propto \exp[-\beta(\sum_i \frac{p_i^2}{2m} + V(\mathbf{R}))]$ as its unique stationary state if and only if the weight functions and amplitudes obey:
 $$w^D(r) = \left[ w^R(r) \right]^2 = \omega^2(r), \qquad \sigma^2 = 2 \gamma k_B T$$
 where a common cutoff envelope is $\omega(r) = (1 - r/r_c)^2$ (or $1 - r/r_c$) for $r < r_c$ and $0$ for $r \ge r_c$.

---

### Integrating the DPD Equations of Motion
In the standard leapfrog / Verlet algorithm:

$$\mathbf{v}(t + \Delta t/2) = \mathbf{v}(t - \Delta t/2) + \frac{\Delta t}{m}\mathbf{f}(t), \qquad \mathbf{r}(t + \Delta t) = \mathbf{r}(t) + \Delta t\,\mathbf{v}(t + \Delta t/2)$$

Because the dissipative force $\mathbf{F}_{ij}^D$ depends on the velocities $\mathbf{v}(t)$ at time $t$:

$$\mathbf{v}(t) = \frac{1}{2}\left[ \mathbf{v}(t + \Delta t/2) + \mathbf{v}(t - \Delta t/2) \right]$$

$\mathbf{v}(t + \Delta t/2)$ appears on **both sides** of the velocity update! To preserve time-reversibility and accurate temperature control, we must **solve the coupled equations self-consistently** (or use a Hybrid / Lowe–Andersen / Shardlow scheme to enforce detailed balance).


In [38]:
ws.render_section('concept_langevin_vs_dpd')

### Interactive Explorer: Poiseuille Channel Flow — Global Langevin vs. Momentum-Conserving DPD

Compare the steady-state velocity profile $v_x(y)$ of a fluid driven through a channel of width $2H$ under a global Langevin thermostat (which damps absolute velocity $\mathbf{v}_i$) versus Dissipative Particle Dynamics (which damps only relative velocity $\mathbf{v}_{ij}$).

**Key Takeaway:** Because DPD forces depend only on relative velocities $\mathbf{v}_{ij}$ and obey $\mathbf{F}_{ij} = -\mathbf{F}_{ji}$, DPD is Galilean invariant and reproduces the exact parabolic Navier–Stokes velocity profile. A global Langevin thermostat destroys local momentum conservation and flattens the parabolic profile into unphysical Brinkman plug flow.

In [39]:
ws.render_section('q28_dpd_momentum_conservation_reason')

**Q28 · easy** · 💡

Why does Dissipative Particle Dynamics (DPD) rigorously conserve both total linear momentum and total angular momentum, unlike a standard Langevin thermostat?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the symmetry $\mathbf{F}_{ij} = -\mathbf{F}_{ji}$ and the direction $\hat{\mathbf{r}}_{ij}$ of the DPD forces.

</div>
</details>

- [ ] **(A)** Because DPD solves the Schrödinger equation for explicit solvent electrons.
- [ ] **(B)** Because DPD sets both the friction coefficient and the random force to zero whenever particles move.
- [ ] **(C)** Because DPD rescales all particle velocities globally every 100 steps to zero out the total momentum.
- [ ] **(D)** Because DPD applies equal-and-opposite friction and random forces ($\mathbf{F}_{ij} = -\mathbf{F}_{ji}$) to each interacting pair along their central line of centers $\hat{\mathbf{r}}_{ij}$, depending only on relative velocities $\mathbf{v}_{ij}$.

In [40]:
ws.render_section('q29_langevin_hydrodynamic_screening_artifact')

**Q29 · medium** · 💡

What unphysical hydrodynamic artifact occurs if you use a global Langevin thermostat ($-\zeta\mathbf{v}_i$) to simulate pressure-driven Poiseuille flow or vortex relaxation in a fluid?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Inspect how damping absolute velocity $\mathbf{v}_i$ affects Galilean invariance and the parabolic Poiseuille profile in the interactive explorer.

</div>
</details>

- [ ] **(A)** The drag $-\zeta \mathbf{v}_i$ acts on absolute laboratory velocity, breaking Galilean invariance, screening out hydrodynamic momentum fluxes, and flattening the parabolic Navier–Stokes profile into plug flow.
- [ ] **(B)** The fluid viscosity drops to zero and the flow accelerates to infinite velocity.
- [ ] **(C)** Total angular momentum increases exponentially while linear momentum remains exactly conserved.
- [ ] **(D)** The fluid spontaneously crystallizes into a body-centered cubic solid at any temperature.

In [41]:
ws.render_section('q30_dpd_weight_function_fdt')

**Q30 · medium** · 💡

In DPD, the pairwise dissipative force has mean $\langle \zeta_{ij} \rangle = -\gamma w^D(r_{ij})(\mathbf{v}_{ij}\cdot\hat{\mathbf{r}}_{ij})$ and the random force has variance $\langle \zeta_{ij}^2 \rangle = \sigma^2 [w^R(r_{ij})]^2$. What relation between the spatial weight functions $w^D(r)$ and $w^R(r)$ and amplitudes $\gamma, \sigma$ is required so that the steady state is the exact canonical Boltzmann distribution?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the Español–Warren DPD fluctuation-dissipation condition on the 'Dissipative Particle Dynamics (DPD)' slide.

</div>
</details>

- [ ] **(A)** $w^D(r) = w^R(r)$ and $\sigma = \gamma k_B T$
- [ ] **(B)** $w^D(r) = [w^R(r)]^2$ and $\sigma^2 = 2\gamma k_B T$
- [ ] **(C)** $w^D(r) = 1 / w^R(r)$ and $\sigma^2 = \gamma / (2 k_B T)$
- [ ] **(D)** $w^D(r) = 0$ and $w^R(r) = (1 - r/r_c)^2$

In [42]:
ws.render_section('q31_dpd_noise_amplitude_numeric')

**Q31 · medium** · 💡

A DPD simulation is parameterized with pairwise friction coefficient $\gamma = 4.5$ and thermal energy $k_B T = 1.0$ (in reduced DPD units). According to the DPD fluctuation–dissipation theorem $\sigma^2 = 2 \gamma k_B T$, what must the random force noise amplitude $\sigma$ be set to?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Solve the Español-Warren fluctuation-dissipation relation for the noise amplitude $\sigma$.

</div>
</details>

_Numeric answer required._

In [43]:
ws.render_section('q32_dpd_cutoff_weight_numeric')

**Q32 · easy** · 💡

Using the randomized force cutoff function $\omega(r) = (1 - r/r_c)^2$ for $r < r_c$  with $r_c = 1.0$ and $k_B T = 1.5$, calculate the variance $\langle \zeta^2 \rangle = 2 k_B T \omega^2(r_{ij})$ for a pair of DPD beads separated by $r_{ij} = 0.50$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the spatial weight function at the specified separation and scale by $2 k_B T$.

</div>
</details>

_Numeric answer required._

In [44]:
ws.render_section('q33_dpd_verlet_self_consistent')

**Q33 · medium** · 💡

Why can't the standard explicit Verlet leapfrog step $\mathbf{v}(t + \Delta t/2) = \mathbf{v}(t - \Delta t/2) + \Delta t\,\mathbf{f}(t)/m$ be evaluated in a single direct pass in Dissipative Particle Dynamics?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check what $\mathbf{f}(t)$ depends on in DPD.

</div>
</details>

- [ ] **(A)** Because the particle masses $m_i$ change randomly at every half-step.
- [ ] **(B)** Because the conservative force $\mathbf{f}^C(\mathbf{r})$ depends on the third time derivative of position (jerk).
- [ ] **(C)** Because the dissipative force $\mathbf{f}(t)$ depends on the instantaneous velocity $\mathbf{v}(t) = \frac{1}{2}[\mathbf{v}(t + \Delta t/2) + \mathbf{v}(t - \Delta t/2)]$, so $\mathbf{v}(t + \Delta t/2)$ appears implicitly on both sides and must be solved self-consistently.
- [ ] **(D)** Because Verlet integration can only be applied to 1D lattice gases.

In [45]:
ws.render_section('q34_dpd_pair_dissipative_force_sign')

**Q34 · medium** · 💡

In DPD, suppose two particles $i$ and $j$ within $r_{ij} < r_c$ are moving directly *toward* each other along their line of centers, so that $(\mathbf{v}_i - \mathbf{v}_j)\cdot(\mathbf{r}_i - \mathbf{r}_j) < 0$ (i.e., $\mathbf{v}_{ij}\cdot\hat{\mathbf{r}}_{ij} < 0$). In what direction does the dissipative force $\mathbf{F}_{ij}^D = -\gamma w^D(r_{ij})(\mathbf{v}_{ij}\cdot\hat{\mathbf{r}}_{ij})\hat{\mathbf{r}}_{ij}$ act on particle $i$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the sign of $-\gamma w^D(r_{ij})(\mathbf{v}_{ij}\cdot\hat{\mathbf{r}}_{ij})$ when $\mathbf{v}_{ij}\cdot\hat{\mathbf{r}}_{ij} < 0$.

</div>
</details>

- [ ] **(A)** The dissipative force is identically zero whenever two particles move toward each other.
- [ ] **(B)** Along $-\hat{\mathbf{r}}_{ij}$ (attractive, accelerating particle $i$ toward particle $j$).
- [ ] **(C)** Perpendicular to $\hat{\mathbf{r}}_{ij}$, causing the pair to rotate around their center of mass.
- [ ] **(D)** Along $+\hat{\mathbf{r}}_{ij}$ (repulsive, pushing particle $i$ away from particle $j$ to oppose their relative approach).

In [46]:
ws.render_section('q35_dpd_pure_rotation_dissipation')

**Q35 · medium** · 💡

Two DPD particles $i$ and $j$ separated by $r_{ij} = 0.6 r_c$ undergo rigid-body rotation around their midpoint so that their relative velocity $\mathbf{v}_{ij} = \mathbf{v}_i - \mathbf{v}_j$ (with magnitude $|\mathbf{v}_{ij}| = 5.0$) is strictly perpendicular to the interparticle unit vector $\hat{\mathbf{r}}_{ij}$ ($\mathbf{v}_{ij} \perp \hat{\mathbf{r}}_{ij}$). What is the magnitude $|\mathbf{F}_{ij}^D|$ of the DPD dissipative force between them?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the dot product $\mathbf{v}_{ij} \cdot \hat{\mathbf{r}}_{ij}$ when $\mathbf{v}_{ij} \perp \hat{\mathbf{r}}_{ij}$.

</div>
</details>

_Numeric answer required._

In [47]:
ws.render_section('sec_synthesis_comparison')

### Summary Comparison Across Dynamics & Sampling Methods

| Method | Momentum $\mathbf{p}_i$ Retained? | Local Momentum Conserved? | Hydrodynamics Captured? | Scaling per Step | Primary Use Case |
|:--- |:---: |:---: |:---: |:---: |:--- |
| **All-Atom MD** | Yes | Yes | Yes (explicit solvent) | $\mathcal{O}(N_{\text{all}})$ | Femtosecond–nanosecond atomic trajectories |
| **Langevin Dynamics** | Yes | **No** ($-\zeta\mathbf{p}_i$ global drag) | **No** (screened) | $\mathcal{O}(N_{\text{solute}})$ | NVT thermostatting & single-molecule inertial barrier crossing |
| **Free-Draining Brownian** | No (overdamped) | **No** | **No** | $\mathcal{O}(N_{\text{solute}})$ | Overdamped diffusion where hydrodynamics is negligible |
| **Brownian + Oseen** | No (overdamped) | Implicit via $\underline{\mathbf{D}}_{ij}$ | Yes (linear $1/r$ Stokes) | **$\mathcal{O}(N_{\text{solute}}^3)$** | Dilute polymers/colloids needing far-field Stokes coupling |
| **Hybrid Smart MC** | No | No | Optional | $\mathcal{O}(N_{\text{solute}})$ | Exact canonical Boltzmann sampling at arbitrary $\tau$ |
| **DPD** | Yes | **Yes** ($\mathbf{F}_{ij} = -\mathbf{F}_{ji}$) | **Yes** (Navier–Stokes) | $\mathcal{O}(N_{\text{beads}})$ | Mesoscale complex fluids, shear flow, vesicles, block copolymers |


In [48]:
ws.render_section('q36_method_selection_hydrodynamic_shear')

**Q36 · medium** · 💡

You want to simulate $N = 10^5$ coarse-grained polymer beads undergoing shear flow in a microchannel where preserving Navier–Stokes hydrodynamics is critical, and an $\mathcal{O}(N^3)$ Cholesky decomposition is computationally prohibitive. Which method from this lesson is best suited?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the scaling and momentum conservation of Oseen Brownian Dynamics, Langevin Dynamics, and DPD.

</div>
</details>

- [ ] **(A)** Dissipative Particle Dynamics (DPD), which conserves local momentum via short-range $\mathcal{O}(N)$ pairwise forces and recovers Navier–Stokes hydrodynamics without explicit solvent atoms.
- [ ] **(B)** Brownian Dynamics with dense Oseen tensor Cholesky factorization at every step.
- [ ] **(C)** Global Langevin Dynamics with a large friction coefficient $\zeta$ to damp absolute velocities.
- [ ] **(D)** Metropolis Monte Carlo with non-local cluster flips.

In [49]:
ws.render_section('q37_generalized_trotter_three_operators')

**Q37 · medium** · 💡

According to the Generalized Trotter Formula for three operators $\hat{A}, \hat{B}, \hat{C}$ that are each bounded below, how can the Boltzmann/propagator operator $\hat{\rho} = e^{-\beta(\hat{A} + \hat{B} + \hat{C})}$ be written in terms of $\tau = \beta/n$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Generalized Trotter Formula'.

</div>
</details>

- [ ] **(A)** $\hat{\rho} = e^{-\beta\hat{A}} + e^{-\beta\hat{B}} + e^{-\beta\hat{C}}$ for any finite $\beta$
- [ ] **(B)** $\hat{\rho} = \lim_{n\to\infty} [ e^{-\tau\hat{A}} e^{-\tau\hat{B}} e^{-\tau\hat{C}} ]^n$
- [ ] **(C)** $\hat{\rho} = \lim_{n\to\infty} [ e^{-\beta\hat{A}} + e^{-\beta\hat{B}} + e^{-\beta\hat{C}} ] / n$
- [ ] **(D)** Trotter's theorem only holds for at most two operators and fails for three operators.

In [50]:
ws.render_section('q38_cholesky_operation_ratio_numeric')

**Q38 · easy** · 💡

Because Cholesky decomposition of the $3N \times 3N$ Oseen diffusion tensor scales as $\mathcal{O}(N^3)$, by what factor does the Cholesky CPU cost per step increase if the number of colloidal particles $N$ is quadrupled from $N = 250$ to $N = 1{,}000$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Cube the ratio of the new particle count to the old particle count based on the cubic scaling of dense Cholesky decomposition.

</div>
</details>

_Numeric answer required._

In [51]:
ws.render_section('q39_generalized_langevin_memory')

**Q39 · medium** · 💡

When a solute particle is comparable in size/timescale to the solvent molecules so that the solvent does not relax instantaneously, the white-noise delta correlation $\langle p_{\text{solvent}}(t)p_{\text{solvent}}(t') \rangle = 2mk_B T\zeta\delta(t-t')$ is replaced by a time-dependent memory function $\langle p_{\text{solvent}}(t)p_{\text{solvent}}(t') \rangle = C(t - t')$. What physical feature does this non-Markovian correlation represent?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about time-delayed feedback from the surrounding solvent.

</div>
</details>

- [ ] **(A)** Instantaneous quantum tunneling of the heavy solute across the simulation box.
- [ ] **(B)** A breakdown of energy conservation that heats the system to infinite temperature.
- [ ] **(C)** Time-delayed solvent memory effects (viscoelastic/hydrodynamic history where past motion of the solute continues to exert correlated forces at later times).
- [ ] **(D)** Elimination of all conservative interatomic forces $\mathbf{F}(\mathbf{r})$.

In [52]:
ws.render_section('q40_overall_method_properties_multiselect')

**Q40 · hard** · 💡

Which of the following statements accurately describe Brownian Dynamics, Smart Monte Carlo, Langevin Dynamics, and Dissipative Particle Dynamics? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Verify each statement against the Smoluchowski equation, Smart MC acceptance, Oseen Cholesky scaling, and DPD conservation laws.

</div>
</details>

- [ ] **(A)** The zero-flux stationary solution of the Smoluchowski equation is strictly the canonical Boltzmann distribution $\rho_{\text{eq}}(\mathbf{R}) \propto \exp(-\beta V(\mathbf{R}))$.
- [ ] **(B)** In Hybrid Brownian / Smart MC, enforcing Metropolis rejections guarantees exact canonical static averages for any step size $\tau$, while accurate dynamics requires an acceptance ratio near 1 ($\sim 99\%$).
- [ ] **(C)** In DPD, pairwise random and dissipative forces act along the central vector $\hat{\mathbf{r}}_{ij}$ with $\mathbf{F}_{ij} = -\mathbf{F}_{ji}$, conserving both linear and angular momentum.
- [ ] **(D)** A global Langevin thermostat strictly conserves local fluid momentum and leaves parabolic Poiseuille channel flow completely unaltered at high $\zeta$.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()